In [15]:
#Creating the tools

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
#An API wrapper is basically a Python class that makes it easier to communicate with an external API/service.
"""

API Wrapper
     ↓
provides access to an external service

Tool
     ↓
makes that capability usable by an Agent

"""

'\n\nAPI Wrapper\n     ↓\nprovides access to an external service\n\nTool\n     ↓\nmakes that capability usable by an Agent\n\n'

In [16]:
#Tool-1: ARxiv

arxiv_api_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)
arxiv.name

'arxiv'

In [17]:
#Tool-1: wikipedia
wikipedia_api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wikipedia = WikipediaQueryRun(api_wrapper=wikipedia_api_wrapper)
wikipedia.name

'wikipedia'

In [18]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings


os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9389.96it/s]


In [20]:

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
final_documents = text_splitter.split_documents(documents=docs)
vectorstore = FAISS.from_documents(final_documents,embedding=embeddings)
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3272d1090>, search_kwargs={})

Website
   ↓
WebBaseLoader
   ↓
Documents
   ↓
Text Splitter
   ↓
Chunks
   ↓
Embeddings
   ↓
Vectors
   ↓
FAISS
   ↓
Retriever
   ↓
Tool
   ↓
Agent

In [21]:
#tool - 3
from langchain_classic.tools.retriever import create_retriever_tool
langsmith_search_tool = create_retriever_tool(retriever,"LangSmith-Search","Search any information about Langsmith")
langsmith_search_tool      #This is where your RAG system becomes an Agent Tool.

StructuredTool(name='LangSmith-Search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x300ada770>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x300ada8d0>)

In [22]:
#all tools
tools = [arxiv,wikipedia,langsmith_search_tool]

In [ ]:
#Prompt template

from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent",
    include_model=False,
    dangerously_pull_public_prompt=True,
)
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='Syst

In [30]:
""""

* Think of Client as a Python object that allows your application to communicate with LangSmith's server.

Your Python program
        │
        │ LangSmith Client
        ▼
    LangSmith
        │
        ├── Prompts
        ├── Traces
        ├── Datasets
        ├── Evaluations
        └── Projects


        In this code, we're interested in prompts.

Owner: hwchase17
Prompt: openai-functions-agent

This is a pre-built prompt designed for an OpenAI-functions-style agent.

Instead of manually writing something like:

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad")
])

"""

'"\n\n* Think of Client as a Python object that allows your application to communicate with LangSmith\'s server.\n\nYour Python program\n        │\n        │ LangSmith Client\n        ▼\n    LangSmith\n        │\n        ├── Prompts\n        ├── Traces\n        ├── Datasets\n        ├── Evaluations\n        └── Projects\n\n\n        In this code, we\'re interested in prompts.\n\nOwner: hwchase17\nPrompt: openai-functions-agent\n\nThis is a pre-built prompt designed for an OpenAI-functions-style agent.\n\nInstead of manually writing something like:\n\nprompt = ChatPromptTemplate.from_messages([\n    ("system", "You are a helpful assistant."),\n    ("human", "{input}"),\n    MessagesPlaceholder("agent_scratchpad")\n])\n\n'

In [24]:
from langchain_groq import ChatGroq


llm=ChatGroq(model_name="openai/gpt-oss-20b")

In [25]:
#Agent 

from langchain_classic.agents import create_openai_tools_agent, AgentExecutor
agent = create_openai_tools_agent(llm,tools,prompt)
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='

In [27]:
agent_executor.invoke({"input":"What is Langsmith?"})




> Entering new AgentExecutor chain...

Invoking: `LangSmith-Search` with `{'query': 'LangSmith'}`


evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.com (no credit card required).

cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and Lond

{'input': 'What is Langsmith?',
 'output': '**LangSmith** is a platform that provides end‑to‑end observability, debugging, evaluation, and deployment tooling for applications that use large language models (LLMs).  \n\nKey features include:\n\n| Feature | What it does |\n|---------|--------------|\n| **Observability** | Capture detailed traces of every LLM call (prompts, completions, tool calls, etc.) so you can see exactly what happened in production. |\n| **Debugging & Monitoring** | Inspect individual traces, compare runs, and surface failures or performance regressions. |\n| **Evaluation** | Build custom evaluation datasets and run them against your LLMs to quantify quality. |\n| **Prompt Engineering** | Store, version, and share prompt templates and configurations. |\n| **Deployment** | Integrate with popular frameworks (OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, etc.) and deploy models at scale. |\n| **Hybrid & Self‑hosted Options** | Run LangSmith on the cloud, in a 

In [ ]:
agent_executor.invoke({"input":"What is Machine Learning?"})




> Entering new AgentExecutor chain...
**Machine Learning (ML)** is a subfield of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to perform tasks without being explicitly programmed. Instead of hard‑coding rules, ML systems learn patterns from data and improve their performance over time.

---

### Core Concepts

| Concept | What it means |
|---------|---------------|
| **Training data** | A collection of examples (inputs and desired outputs) that the algorithm uses to learn. |
| **Model** | The mathematical representation (e.g., a neural network, decision tree, support vector machine) that maps inputs to outputs. |
| **Learning algorithm** | The procedure that adjusts the model’s parameters based on the training data (e.g., gradient descent). |
| **Loss function** | A metric that quantifies how far the model’s predictions are from the true values; the learning algorithm seeks to minimize it. |
| **Evaluation** | Testing

{'input': 'What is Machine Learning?',
 'output': '**Machine Learning (ML)** is a subfield of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to perform tasks without being explicitly programmed. Instead of hard‑coding rules, ML systems learn patterns from data and improve their performance over time.\n\n---\n\n### Core Concepts\n\n| Concept | What it means |\n|---------|---------------|\n| **Training data** | A collection of examples (inputs and desired outputs) that the algorithm uses to learn. |\n| **Model** | The mathematical representation (e.g., a neural network, decision tree, support vector machine) that maps inputs to outputs. |\n| **Learning algorithm** | The procedure that adjusts the model’s parameters based on the training data (e.g., gradient descent). |\n| **Loss function** | A metric that quantifies how far the model’s predictions are from the true values; the learning algorithm seeks to minimize it. |\n| *

In [ ]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})




> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c**Paper ID:** 1706.03762  
**Title:** *Attention Is All You Need*  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Published:** 2017 (arXiv:1706.03762)

---

### What the paper is about

The paper introduces the **Transformer** architecture—a new neural network model for sequence‑to‑sequence tasks that relies **entirely on attention mechanisms** and dispenses with recurrent (RNN) and convolutional (CNN) layers. The authors show that this design can achieve, and often surpass, state‑of‑the‑art performance on machine‑translation benchmarks while being far more pa

{'input': "What's the paper 1706.03762 about?",
 'output': '**Paper ID:** 1706.03762  \n**Title:** *Attention Is All You Need*  \n**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob\u202fUszkoreit, Llion\u202fJones, Aidan\u202fN.\u202fGomez, Łukasz\u202fKaiser, Illia\u202fPolosukhin  \n**Published:** 2017 (arXiv:1706.03762)\n\n---\n\n### What the paper is about\n\nThe paper introduces the **Transformer** architecture—a new neural network model for sequence‑to‑sequence tasks that relies **entirely on attention mechanisms** and dispenses with recurrent (RNN) and convolutional (CNN) layers. The authors show that this design can achieve, and often surpass, state‑of‑the‑art performance on machine‑translation benchmarks while being far more parallelizable and efficient to train.\n\n---\n\n### Key ideas and contributions\n\n| Concept | What it is | Why it matters |\n|---------|------------|----------------|\n| **Self‑attention** | Each token in a sequence attends to every other toke